# CMSC848G Project: Deep Learning with PyTorch

CIFAR-10 image classification experiments covering:
- **Part 1**: Adapting the PyTorch MNIST CNN to CIFAR-10
- **Part 2**: VGG11 ablation study (batch norm, activations, optimizer, init, dropout)


[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/jumeike/cmsc848g-project/blob/main/notebook.ipynb)

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torchvision import datasets, transforms, models
from torch.optim.lr_scheduler import StepLR
import matplotlib.pyplot as plt

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

CIFAR10_MEAN = (0.4914, 0.4822, 0.4465)
CIFAR10_STD  = (0.2023, 0.1994, 0.2010)

transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(CIFAR10_MEAN, CIFAR10_STD),
])

train_set = datasets.CIFAR10('./data', train=True,  download=True, transform=transform)
test_set  = datasets.CIFAR10('./data', train=False, download=True, transform=transform)
print('Dataset ready.')

---
## Part 1: Adapting the MNIST CNN to CIFAR-10

Two changes from the original MNIST example:
1. `Conv2d(1, 32, 3, 1)` → `Conv2d(3, 32, 3, 1)` — RGB input
2. `Linear(9216, 128)` → `Linear(12544, 128)` — larger spatial feature map (32×32 → 14×14 after pool)

In [ ]:
class Net(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv1    = nn.Conv2d(3, 32, 3, 1)   # 3-channel RGB
        self.conv2    = nn.Conv2d(32, 64, 3, 1)
        self.dropout1 = nn.Dropout(0.25)
        self.dropout2 = nn.Dropout(0.5)
        self.fc1      = nn.Linear(12544, 128)     # 64 * 14 * 14
        self.fc2      = nn.Linear(128, 10)

    def forward(self, x):
        x = F.relu(self.conv1(x))
        x = F.relu(self.conv2(x))
        x = F.max_pool2d(x, 2)
        x = self.dropout1(x)
        x = torch.flatten(x, 1)
        x = F.relu(self.fc1(x))
        x = self.dropout2(x)
        return F.log_softmax(self.fc2(x), dim=1)

In [ ]:
EPOCHS_P1   = 14
LOG_INTERVAL = 100

train_loader_p1 = torch.utils.data.DataLoader(train_set, batch_size=64,  shuffle=True,  num_workers=2, pin_memory=True)
test_loader_p1  = torch.utils.data.DataLoader(test_set,  batch_size=1000, shuffle=False, num_workers=2, pin_memory=True)

torch.manual_seed(1)
model_p1  = Net().to(device)
optimizer = optim.Adadelta(model_p1.parameters(), lr=1.0)
scheduler = StepLR(optimizer, step_size=1, gamma=0.7)

p1_losses, p1_accs = [], []

for epoch in range(1, EPOCHS_P1 + 1):
    model_p1.train()
    epoch_loss = 0
    for batch_idx, (data, target) in enumerate(train_loader_p1):
        data, target = data.to(device), target.to(device)
        optimizer.zero_grad()
        loss = F.nll_loss(model_p1(data), target)
        loss.backward()
        optimizer.step()
        epoch_loss += loss.item()
    scheduler.step()

    model_p1.eval()
    correct = 0
    with torch.no_grad():
        for data, target in test_loader_p1:
            data, target = data.to(device), target.to(device)
            pred = model_p1(data).argmax(dim=1)
            correct += pred.eq(target).sum().item()
    acc = 100. * correct / len(test_set)
    avg_loss = epoch_loss / len(train_loader_p1)
    p1_losses.append(avg_loss)
    p1_accs.append(acc)
    print(f'Epoch {epoch:2d}  Loss: {avg_loss:.4f}  Accuracy: {acc:.2f}%')

In [ ]:
epochs = range(1, EPOCHS_P1 + 1)
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 4))
ax1.plot(epochs, p1_losses, marker='o'); ax1.set_title('Train Loss'); ax1.set_xlabel('Epoch'); ax1.grid(True, alpha=0.4)
ax2.plot(epochs, p1_accs,   marker='o', color='orange'); ax2.set_title('Test Accuracy (%)'); ax2.set_xlabel('Epoch'); ax2.grid(True, alpha=0.4)
plt.suptitle(f'Part 1 - Final accuracy: {p1_accs[-1]:.1f}%', fontweight='bold')
plt.tight_layout()
plt.show()

---
## Part 2: VGG11 on CIFAR-10

VGG11 from `torchvision` adapted for CIFAR-10:
- Final classifier: `Linear(4096, 1000)` → `Linear(4096, 10)`

All experiments use Adam (lr=2e-4, betas=(0.5, 0.999)), batch size 128, **100 epochs**.

In [ ]:
def replace_relu(model):
    for name, module in model.named_children():
        if isinstance(module, nn.ReLU):
            setattr(model, name, nn.LeakyReLU(inplace=True))
        else:
            replace_relu(module)

def disable_dropout(model):
    for name, module in model.named_children():
        if isinstance(module, nn.Dropout):
            setattr(model, name, nn.Identity())
        else:
            disable_dropout(module)

def init_weights(model, init_type):
    for m in model.modules():
        if isinstance(m, nn.Conv2d):
            if init_type == 'kaiming':
                nn.init.kaiming_normal_(m.weight, mode='fan_out', nonlinearity='relu')
            else:
                nn.init.xavier_uniform_(m.weight)
            if m.bias is not None:
                nn.init.constant_(m.bias, 0)

def build_vgg(variant='vgg11', activation='relu', init='kaiming', no_dropout=False):
    model = models.vgg11_bn(weights=None) if variant == 'vgg11_bn' else models.vgg11(weights=None)
    model.classifier[6] = nn.Linear(4096, 10)
    if activation == 'leaky_relu':
        replace_relu(model)
    if no_dropout:
        disable_dropout(model)
    init_weights(model, init)
    return model.to(device)

train_loader_p2 = torch.utils.data.DataLoader(train_set, batch_size=128, shuffle=True,  num_workers=2, pin_memory=True)
test_loader_p2  = torch.utils.data.DataLoader(test_set,  batch_size=1000, shuffle=False, num_workers=2, pin_memory=True)

EPOCHS_P2 = 100

def run_experiment(label, model, optimizer):
    print(f'\n=== {label} ===')
    losses, accs = [], []
    for epoch in range(1, EPOCHS_P2 + 1):
        model.train()
        total = 0
        for data, target in train_loader_p2:
            data, target = data.to(device), target.to(device)
            optimizer.zero_grad()
            loss = F.cross_entropy(model(data), target)
            loss.backward()
            optimizer.step()
            total += loss.item()
        model.eval()
        correct = 0
        with torch.no_grad():
            for data, target in test_loader_p2:
                data, target = data.to(device), target.to(device)
                correct += model(data).argmax(1).eq(target).sum().item()
        acc = 100. * correct / len(test_set)
        avg = total / len(train_loader_p2)
        losses.append(avg)
        accs.append(acc)
        print(f'  Epoch {epoch:2d}  Loss: {avg:.4f}  Accuracy: {acc:.2f}%')
    return losses, accs

print('Helpers ready.')

### Section 1: Model Components
#### 1a: VGG11, ReLU (baseline)

In [ ]:
torch.manual_seed(1)
m1a = build_vgg('vgg11', activation='relu', init='kaiming')
losses_1a, accs_1a = run_experiment('1a: VGG11, ReLU', m1a,
    optim.Adam(m1a.parameters(), lr=2e-4, betas=(0.5, 0.999)))

#### 1b: VGG11-BN, ReLU

In [ ]:
torch.manual_seed(1)
m1b = build_vgg('vgg11_bn', activation='relu', init='kaiming')
losses_1b, accs_1b = run_experiment('1b: VGG11-BN, ReLU', m1b,
    optim.Adam(m1b.parameters(), lr=2e-4, betas=(0.5, 0.999)))

#### 1c: VGG11, LeakyReLU

In [ ]:
torch.manual_seed(1)
m1c = build_vgg('vgg11', activation='leaky_relu', init='kaiming')
losses_1c, accs_1c = run_experiment('1c: VGG11, LeakyReLU', m1c,
    optim.Adam(m1c.parameters(), lr=2e-4, betas=(0.5, 0.999)))

In [ ]:
ep = range(1, EPOCHS_P2 + 1)
fig, axes = plt.subplots(2, 3, figsize=(14, 7))
for i, (label, losses, accs) in enumerate([
    ('1a: VGG11, ReLU', losses_1a, accs_1a),
    ('1b: VGG11-BN, ReLU', losses_1b, accs_1b),
    ('1c: VGG11, LeakyReLU', losses_1c, accs_1c),
]):
    axes[0,i].plot(ep, losses, marker='o', markersize=2)
    axes[0,i].set_title(label, fontweight='bold'); axes[0,i].set_xlabel('Epoch'); axes[0,i].set_ylabel('Train Loss'); axes[0,i].grid(True, alpha=0.4)
    axes[1,i].plot(ep, accs, marker='o', markersize=2, color='orange')
    axes[1,i].set_title(label, fontweight='bold'); axes[1,i].set_xlabel('Epoch'); axes[1,i].set_ylabel('Test Accuracy (%)'); axes[1,i].grid(True, alpha=0.4)
plt.tight_layout()
plt.show()

### Section 2: Training Methods
Base: VGG11 + LeakyReLU (1c). One change at a time.

#### 2a: SGD optimizer (no momentum, lr=2e-4)

In [ ]:
torch.manual_seed(1)
m2a = build_vgg('vgg11', activation='leaky_relu', init='kaiming')
losses_2a, accs_2a = run_experiment('2a: LeakyReLU + SGD', m2a,
    optim.SGD(m2a.parameters(), lr=2e-4))

#### 2b: Batch size 256

In [ ]:
train_loader_bs256 = torch.utils.data.DataLoader(train_set, batch_size=256, shuffle=True, num_workers=2, pin_memory=True)

torch.manual_seed(1)
m2b = build_vgg('vgg11', activation='leaky_relu', init='kaiming')
opt2b = optim.Adam(m2b.parameters(), lr=2e-4, betas=(0.5, 0.999))

print('\n=== 2b: LeakyReLU + BS=256 ===')
losses_2b, accs_2b = [], []
for epoch in range(1, EPOCHS_P2 + 1):
    m2b.train()
    total = 0
    for data, target in train_loader_bs256:
        data, target = data.to(device), target.to(device)
        opt2b.zero_grad()
        loss = F.cross_entropy(m2b(data), target)
        loss.backward(); opt2b.step()
        total += loss.item()
    m2b.eval()
    correct = 0
    with torch.no_grad():
        for data, target in test_loader_p2:
            data, target = data.to(device), target.to(device)
            correct += m2b(data).argmax(1).eq(target).sum().item()
    acc = 100. * correct / len(test_set)
    avg = total / len(train_loader_bs256)
    losses_2b.append(avg); accs_2b.append(acc)
    print(f'  Epoch {epoch:2d}  Loss: {avg:.4f}  Accuracy: {acc:.2f}%')

#### 2c: Xavier initialization (uses VGG11-BN for stability)

In [ ]:
torch.manual_seed(1)
m2c = build_vgg('vgg11_bn', activation='leaky_relu', init='xavier')
losses_2c, accs_2c = run_experiment('2c: LeakyReLU + Xavier', m2c,
    optim.Adam(m2c.parameters(), lr=2e-4, betas=(0.5, 0.999)))

#### 2d: No dropout

In [ ]:
torch.manual_seed(1)
m2d = build_vgg('vgg11', activation='leaky_relu', init='kaiming', no_dropout=True)
losses_2d, accs_2d = run_experiment('2d: LeakyReLU, No Dropout', m2d,
    optim.Adam(m2d.parameters(), lr=2e-4, betas=(0.5, 0.999)))

In [ ]:
ep = range(1, EPOCHS_P2 + 1)
fig, axes = plt.subplots(2, 4, figsize=(18, 7))
for i, (label, losses, accs) in enumerate([
    ('2a: SGD', losses_2a, accs_2a),
    ('2b: BS=256', losses_2b, accs_2b),
    ('2c: Xavier', losses_2c, accs_2c),
    ('2d: No Dropout', losses_2d, accs_2d),
]):
    axes[0,i].plot(ep, losses, marker='o', markersize=2)
    axes[0,i].set_title(label, fontweight='bold'); axes[0,i].set_xlabel('Epoch'); axes[0,i].set_ylabel('Train Loss'); axes[0,i].grid(True, alpha=0.4)
    axes[1,i].plot(ep, accs, marker='o', markersize=2, color='orange')
    axes[1,i].set_title(label, fontweight='bold'); axes[1,i].set_xlabel('Epoch'); axes[1,i].set_ylabel('Test Accuracy (%)'); axes[1,i].grid(True, alpha=0.4)
plt.tight_layout()
plt.show()

### Summary: Final Accuracy Comparison

In [ ]:
import matplotlib.ticker as ticker

labels  = ['1a: VGG11\nReLU', '1b: VGG11-BN\nReLU', '1c: VGG11\nLeakyReLU',
           '2a: SGD', '2b: BS=256', '2c: Xavier', '2d: No Dropout']
results = [accs_1a[-1], accs_1b[-1], accs_1c[-1], accs_2a[-1], accs_2b[-1], accs_2c[-1], accs_2d[-1]]

x = range(len(labels))
fig, ax = plt.subplots(figsize=(13, 5))
bars = ax.bar(x, results, width=0.5, color='steelblue', edgecolor='black', linewidth=0.8)
for bar in bars:
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.3,
            f'{bar.get_height():.1f}%', ha='center', va='bottom', fontsize=10)
ax.set_xticks(list(x))
ax.set_xticklabels(labels, fontsize=10)
ax.set_ylabel('Test Accuracy (%)', fontsize=11)
ax.set_title(f'Part 2 Final Test Accuracy ({EPOCHS_P2} epochs)', fontweight='bold')
ax.yaxis.set_major_locator(ticker.MultipleLocator(10))
for spine in ax.spines.values():
    spine.set_linewidth(1.2)
ax.grid(axis='y', alpha=0.3)
ax.set_axisbelow(True)
plt.tight_layout()
plt.show()